# AGILAB API first run in Colab (Source Checkout)

This notebook shows the smallest source-checkout AGILAB API path from Google Colab.
It clones the repository, installs AGILAB in editable mode, and runs the built-in
`mycode_project` locally through `AgiEnv` and `AGI.run(...)`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_first_run_source.ipynb)


In [ ]:
# Source-checkout Colab path: clone AGILAB and install editable packages.
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/agilab")
STATE_FILE = Path("/content/.agilab_colab_source_state.json")
STATE_VERSION = 1

if (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_ROOT, check=True)
else:
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ThalesGroup/agilab.git", str(REPO_ROOT)], check=True)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
state = {}
if STATE_FILE.is_file():
    try:
        state = json.loads(STATE_FILE.read_text())
    except Exception:
        state = {}

needs_install = (
    state.get("version") != STATE_VERSION
    or state.get("mode") != "source"
    or state.get("head") != head
)

if needs_install:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "agilab", "agi-core", "agi-cluster", "agi-node", "agi-env"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir",
        "-e", "/content/agilab/src/agilab/core/agi-env",
        "-e", "/content/agilab/src/agilab/core/agi-node",
        "-e", "/content/agilab/src/agilab/core/agi-cluster",
        "-e", "/content/agilab/src/agilab/core/agi-core",
        "-e", "/content/agilab",
    ], check=True)
    STATE_FILE.write_text(json.dumps({"version": STATE_VERSION, "mode": "source", "head": head, "install_pid": os.getpid()}))
    raise SystemExit("AGILAB source packages installed or updated. Restart the Colab runtime, then rerun the notebook from the top.")

print("AGILAB source packages already prepared for this Colab runtime.")


In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

STATE_FILE = Path("/content/.agilab_colab_source_state.json")
if STATE_FILE.is_file():
    try:
        _colab_state = json.loads(STATE_FILE.read_text())
    except Exception:
        _colab_state = {}
    if _colab_state.get("install_pid") == os.getpid():
        raise RuntimeError("AGILAB source packages were installed in this same Colab kernel. Restart the runtime, then rerun the notebook from the top.")

os.environ["IS_SOURCE_ENV"] = "1"
os.environ["AGI_CLUSTER_ENABLED"] = "0"
os.environ.pop("IS_WORKER_ENV", None)

for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.", "numpy", "numpy.", "scipy", "scipy.", "pandas", "pandas.", "polars", "polars.", "numba", "numba.")):
        del sys.modules[name]
importlib.invalidate_caches()

REPO_ROOT = Path("/content/agilab")
for entry in reversed([
    REPO_ROOT / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-core" / "src",
]):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

from agi_cluster.agi_distributor import AGI
from agi_env import AgiEnv

REPO_ROOT = Path("/content/agilab")
APPS_PATH = REPO_ROOT / "src" / "agilab" / "apps"
BUILTIN_ROOT = APPS_PATH / "builtin"
APP = "mycode_project"
LOG_ROOT = Path.home() / "log" / "execute" / "mycode"

def worker_env_ready(app_env):
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        return False
    cmd = ["uv", "--quiet", "run", "--no-sync", "--project", str(worker_venv.parent)]
    pyvers_worker = getattr(app_env, "pyvers_worker", None)
    if pyvers_worker:
        cmd.extend(["--python", str(pyvers_worker)])
    cmd.extend(["python", "-c", "import agi_env, agi_node"])
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    return result.returncode == 0

async def install_if_needed(app_env, *, scheduler="127.0.0.1", workers=None, modes_enabled=0):
    if workers is None:
        workers = {"127.0.0.1": 1}
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if worker_env_ready(app_env):
        return False
    action = "Installing"
    if worker_venv.parent.exists():
        shutil.rmtree(worker_venv.parent, ignore_errors=True)
        action = "Reinstalling"
    print(f"{action} worker for {app_env.app}...")
    await AGI.install(app_env, scheduler=scheduler, workers=workers, modes_enabled=modes_enabled)
    return True

print("Repo root:", REPO_ROOT)
print("Apps path:", APPS_PATH)
print("Builtin root:", BUILTIN_ROOT)
print("App:", APP)
print("Log root:", LOG_ROOT)


In [ ]:
app_env = AgiEnv(apps_path=APPS_PATH, app=APP, verbose=0)
await install_if_needed(app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
result = await AGI.run(
    app_env,
    scheduler="127.0.0.1",
    workers={"127.0.0.1": 1},
    mode=0,
)
result
